# 03 · Extension: SAMME (from scratch)
Report section: *Extension method(s)* + part of *Results*. Runs SAMME on the leakage-safe split and benchmarks it against the betting market (tasks D2-run, D4).

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data.pipeline import build_dataset
from src.data.odds import method_market, market_benchmark
from src.extension import SAMMEClassifier
from src.metrics import accuracy, precision_recall_f1, log_loss
from src.plotting import plot_staged_accuracy, save_fig

## 1. Implementation
SAMME (multi-class AdaBoost) lives in `src/extension/samme.py` and is validated against sklearn in `tests/unit/test_samme.py` (the from-scratch rule: sklearn only validates, never produces a reported number).

## 2. SAMME on the fight data (D2 run)
Fit 200 decision stumps on the symmetrized split (~10 min; the from-scratch stump scans every threshold - see the SAMME-speed backlog item).

In [ ]:
ds = build_dataset(test_frac=0.2, seed=0)
Xtr, ytr = ds.X_train.values, ds.y_train.values
Xte, yte = ds.X_test.values, ds.y_test.values
samme = SAMMEClassifier(n_estimators=200).fit(Xtr, ytr)
pred = samme.predict(Xte)
winner = lambda a: np.array([s.split('-')[0] for s in a])
method = lambda a: np.array([s.split('-')[1] for s in a])
print(f'SAMME  6-class acc = {accuracy(yte, pred):.3f}  macroF1 = {precision_recall_f1(yte, pred)["macro_f1"]:.3f}  logloss = {log_loss(yte, samme.predict_proba(Xte), classes=samme.classes_):.3f}')
print(f'       winner acc = {accuracy(winner(yte), winner(pred)):.3f}   method acc = {accuracy(method(yte), method(pred)):.3f}')
print(f'       (vs LDA baseline: winner 0.630 / method 0.524)')

## 3. Hyperparameter analysis (E2)
**(Milica, issue #14)** placeholder - the helpers are ready below.

In [ ]:
# E2 (Milica, issue #14): hyperparameter sweeps + plots.
# Ready-made helpers:
#   - SAMME n_estimators: ONE fit, then samme.staged_score(Xte, yte) gives accuracy after
#     every round (no refit). Plot with plot_staged_accuracy(scores).
#   - kNN k: loop k, KNNClassifier(k).fit(Xtr,ytr).score(Xte,yte); plot with plot_sweep.
#   - PCA dims: src.baselines.PCA(n_components=d).fit(Xtr); refit models on reduced X.
# Example for the SAMME convergence curve (cheap, reuses the fit above):
scores = samme.staged_score(Xte, yte)
plot_staged_accuracy(scores); save_fig('hyperparam_samme'); plt.show()
print(f'best round: {int(np.argmax(scores))+1}  acc={max(scores):.3f}')

## 4. Odds benchmark (D4)
Compare SAMME against the de-vigged per-method market by log-loss / Brier, on the ORIGINAL-corner test fights that have full market coverage.

In [ ]:
mkt = method_market(ds.df, classes=samme.classes_).loc[ds.test_index]
proba_orig = samme.predict_proba(ds.X_test_orig.values)
res = market_benchmark(ds.y_test_orig.values, proba_orig, mkt.values, samme.classes_)
print(f'fights with market coverage: {res["n_fights"]}')
print(f'log-loss   SAMME = {res["log_loss"]["model"]:.3f}   market = {res["log_loss"]["market"]:.3f}')
print(f'Brier      SAMME = {res["brier"]["model"]:.3f}   market = {res["brier"]["market"]:.3f}')